In [ ]:
import pandas as pd
import pyarrow.feather as feather
import numpy as np
#feather_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step18.ftr"
# "please take the file: glm_data_step2_final.ftr and give it to Gary."

import os as os
#import matplotlib.pyplot as plt
import time as time
import random

#import modin.pandas as pd


In [ ]:
"""1)	Open “superpolicy_final.sas7bdat” . a.	 This file has the superpolicy_id based on the logic Gary wrote in SAS i.	Where is that file (store for completion) 
2)	Read “necessary columns” from step 18 feather file since there will be memory issue otherwise. a.	“necessary columns” = columns needed for objective function, VIN, Date
b.	Merge this data with “superpolicy_final.sas7bdat” using VIN-Date 
    i.	Confirm the number of extra rows after the merge
    1.	Ideally there wont be any extra rows 
        i. Rows were not same. 
3)	Loop to find ideal fits run 500 simulation. 
    a.	For each loop get the 24D vector 
    b.	Calculate the Objective function for train, valid, holdout. 
    c.	If the error is less than SD, then proceed in the loop
    d.	Store at end of loop, seed, Objective function, time taken for each loop 
4)	Plot Loop vs Objective function. 
    Plot Hist of Objective function. 
    a.Choose lowest Objective function ? """




- Open “superpolicy_final.sas7bdat” . a.	 This file has the superpolicy_id based on the logic Gary wrote in SAS i.	Where is that file (store for completion) 

In [ ]:
base_path = 'R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/'

full_feather_file_path = os.path.join(base_path,  "glm_data_step18.ftr")
columns_file_path = os.path.join(base_path,  "columns_glm_data_step2_final_feather.csv") 
superpolicy_file =  os.path.join(base_path, 'superpolicy_final.sas7bdat')

step18_ess_cols = os.path.join(base_path,  "step18_ess_cols.csv")

""" 
feather_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.ftr"

csv_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"
csv_1000row_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv"
csv_allrow_file_path = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final.csv"


file_df1_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df1_1_superpolicy.csv"

file_df2_1_superpolicy = "R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/df2_1_superpolicy.csv" """


In [ ]:
df_a = pd.read_sas( superpolicy_file ,format = 'sas7bdat', encoding="ISO-8859-1")  # read sas7bdat file

In [ ]:
#import modin.config as modin_cfg

#modin_cfg.Engine.put("ray")  # Modin will use Ray
#modin_cfg.Engine.put("dask")  # Modin will use Dask

#modin_cfg.Engine.put('unidist') # Modin will use Unidist
#unidist_cfg.Backend.put('mpi') # Unidist will use MPI backend

In [ ]:
#df_a = pd.DataFrame(df_a_pandas)

In [ ]:
df_a['Date'] = pd.to_datetime(df_a['Date'])
df_a['DateYMD'] = df_a['Date'].dt.strftime('%Y%m%d')
df_a['VIN_Date'] = df_a['VIN'] +'_' +  df_a['DateYMD']


In [ ]:

df_a.drop(columns = ['DateYMD','VIN','Date'], inplace = True)

In [ ]:
df_a = df_a.drop_duplicates( subset = 'VIN_Date', keep = 'first')

In [ ]:
df_a.head()

In [ ]:
#testing to make sure seed is repeatable
if 0:
    SeedNum = 1
    unique_values = df_a['superpolicy_id'].unique().tolist()
    np.random.seed(SeedNum)  
    shuffleunique_values = random.sample(unique_values, len(unique_values))
    temp = sum([a -b for a,b in zip(unique_values, shuffleunique_values) ])

    print(temp)
    unique_values = df_a['superpolicy_id'].unique().tolist()

#Experiment to confirm that random seed is reproducible. 
# Lesson: random seed and np.random seed are different things. 
# Since we are using random.sample since that is the only way to shuffle without altering original copy. 
# np.shuffle was changing original - that is , it was in place. 
if 0:
    for SeedNum in range(1,10):
        random.seed(SeedNum)  
        shuffleunique_values = random.sample(unique_values, len(unique_values))
        temp = sum(abs(np.array(unique_values)-  np.array(shuffleunique_values) ))
        print(temp)
    for SeedNum in range(1,10):
        random.seed(SeedNum)  
        shuffleunique_values = random.sample(unique_values, len(unique_values))
        temp = sum(abs(np.array(unique_values)-  np.array(shuffleunique_values) ))
        print(temp)

In [ ]:
def get_tvh(df_a,SeedNum,sorted_unique_values,tv_split_colname):
    #all rows are assumed train
    # TrainPercentage = 0.6
    ValidPercentage = 0.2
    HoldoutPercentage = 0.2

    #np.random.seed(SeedNum)  
    total_unique = len(unique_values)
    shuffleunique_values = random.sample(sorted_unique_values, total_unique)

    valid_end = int(total_unique * ValidPercentage)
    holdout_end = valid_end +  int(total_unique * HoldoutPercentage)
    

    #train_values = shuffleunique_values[:train_end]
    val_values = shuffleunique_values[:valid_end]
    holdout_values = shuffleunique_values[valid_end:holdout_end]


    value_to_split = {}
    for value in val_values:
        value_to_split[value] = 'valid'
    for value in holdout_values:
        value_to_split[value] = 'holdout'

    df_a[tv_split_colname] = df_a['superpolicy_id'].map(value_to_split)
    return df_a

In [ ]:
def generate_row(df_subset):
    #np.random.seed(seed_num)
    
    agg_columns = ['ee_bi', 'ee_pd', 'ee_pip', 'ee_coll', 'ee_comp','ee_med',
       'incurred_raw_bi', 'incurred_raw_pd', 'incurred_raw_pip', 
       'incurred_raw_coll','incurred_raw_comp', 'incurred_raw_med'] 
    agg_dict = {col: 'sum' for col in agg_columns}
    df_subset_grouped = df_subset.groupby(['vc_Vehicle_Type']).agg(agg_dict).reset_index()

    #df_subset_grouped.head()
    df_subset_grouped['pp_bi'] = df_subset_grouped['incurred_raw_bi']/df_subset_grouped['ee_bi'] 
    df_subset_grouped['pp_pd'] = df_subset_grouped['incurred_raw_pd']/df_subset_grouped['ee_pd'] 
    df_subset_grouped['pp_pip'] = df_subset_grouped['incurred_raw_pip']/df_subset_grouped['ee_pip'] 
    df_subset_grouped['pp_med'] = df_subset_grouped['incurred_raw_med']/df_subset_grouped['ee_med'] 
    df_subset_grouped['pp_coll'] = df_subset_grouped['incurred_raw_coll']/df_subset_grouped['ee_coll'] 
    df_subset_grouped['pp_comp'] = df_subset_grouped['incurred_raw_comp']/df_subset_grouped['ee_comp'] 

    df_subset_grouped.set_index('vc_Vehicle_Type', inplace=True)
    flattened_columns = [f'{idx}_{col}' for idx in df_subset_grouped.index for col in df_subset_grouped.columns]
    flattened_data = df_subset_grouped.values.flatten()

    df_subset_flattened = pd.DataFrame([flattened_data], columns=flattened_columns)
    return df_subset_flattened

In [ ]:
#db_ppee  
#columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp','claim_count_bi', 'claim_count_pd', 'claim_count_pip', 'claim_count_med','claim_count_coll', 'claim_count_comp']
columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp']
df1 = pd.read_feather(full_feather_file_path, columns =  columns_to_load )

#df_a = pd.DataFrame(df_a_pandas)
#write a function 
df1['Date'] = pd.to_datetime(df1['Date'])
df1['DateYMD'] = df1['Date'].dt.strftime('%Y%m%d')
df1['VIN_Date'] = df1['VIN'] +'_' +  df1['DateYMD']

In [ ]:
csv_stat_file = 'df2_1_stat_pp_N500.csv'
stat_file = pd.read_csv(os.path.join(base_path, csv_stat_file))
stat_file['norm_mean_ee'] = stat_file['mean_ee']/np.sum(stat_file['mean_ee'])
stat_file.reset_index()
stat_file

In [ ]:
#df1.to_csv(step18_ess_cols)

In [ ]:
def objectivefunction(train_df_a_pp,stat_file):
    mycol = train_df_a_pp.columns
    stat_file['sample'] = train_df_a_pp[mycol]
    objfunval_train_df = np.sum(np.abs(stat_file['sample'] - stat_file['mean_pp'])*stat_file['norm_mean_ee'])
    return objfunval_train_df


In [ ]:
import multiprocessing
num_processes = multiprocessing.cpu_count()
print(num_processes)

In [ ]:
df_a.columns
df1.head()

In [ ]:
df1.drop(columns = ['DateYMD','VIN','Date'], inplace = True)

In [ ]:
unique_values = df_a['superpolicy_id'].unique()
sorted_unique_values = sorted(unique_values)

tv_split_colname = 'tv_split' #+str(SeedNum)
within_sd_list = []
outof_sd_list = []
pp_columns = stat_file['Unnamed: 0'] # [column for column in stat_file.columns if 'pp' in column]

listtotal_obj_fun  = []
metconditon = 1


In [ ]:

#
df_a = pd.merge(df_a, df1, on = 'VIN_Date', how = 'left')
full_df_a_subset_flattened = generate_row(df_a)



In [ ]:

incurred_columns = [column for column in full_df_a_subset_flattened.columns if 'incurred_raw' in column]
ee_columns = []
pp_columns = []
for i in range(len(incurred_columns)):
    ee_columns.append(incurred_columns[i].replace('incurred_raw','ee'))
    pp_columns.append(incurred_columns[i].replace('incurred_raw','pp'))


In [ ]:
start_time = time.time()

for SeedNum in range(0,1):
    train_df_a_subset_flattened = pd.DataFrame()
    valid_df_a = pd.DataFrame()
    holdout_df_a = pd.DataFrame()

    #get tv_split in df_a
    df_b = get_tvh(df_a,SeedNum,sorted_unique_values,tv_split_colname)
    
    # Get train, text, val 
    #train_df_a =  df_a.loc[df_a['tv_split'] == 'train'].reset_index().drop(['index','tv_split'], axis=1)
    valid_df_a =  df_b.loc[df_a['tv_split'] == 'valid'].reset_index().drop(['index','tv_split'], axis=1)
    holdout_df_a =  df_b.loc[df_a['tv_split'] == 'holdout'].reset_index().drop(['index','tv_split'], axis=1)

    valid_df_a = pd.merge(valid_df_a, df1, on = 'VIN_Date', how = 'left')
    valid_df_a_subset_flattened = generate_row(valid_df_a)
    holdout_df_a = pd.merge(holdout_df_a, df1, on = 'VIN_Date', how = 'left')
    holdout_df_a_subset_flattened = generate_row(holdout_df_a)

    #train_df_a_pp = generate_row_train( valid_df_a_subset_flattened,holdout_df_a_subset_flattened)
    for pp,inc,ee in zip(pp_columns,incurred_columns,ee_columns):
        print(pp,inc,ee)
        train_df_a_subset_flattened[inc] = full_df_a_subset_flattened[inc] - holdout_df_a_subset_flattened[inc] -valid_df_a_subset_flattened[inc]
        train_df_a_subset_flattened[ee] = full_df_a_subset_flattened[ee] - holdout_df_a_subset_flattened[ee] - valid_df_a_subset_flattened[ee]
        train_df_a_subset_flattened[pp] = train_df_a_subset_flattened[inc]/train_df_a_subset_flattened[ee]

    valid_df_a_pp = valid_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    holdout_df_a_pp = holdout_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    #send to objective function 

    train_objval = objectivefunction(train_df_a_subset_flattened,stat_file)
    valid_objval = objectivefunction(valid_df_a_subset_flattened,stat_file)
    holdout_objval = objectivefunction(holdout_df_a_subset_flattened,stat_file)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Elapsed time 3 : {elapsed_time : .2f} seconds")


    # train_objval = total_objval - valid_objval - holdout_objval

    total_obj_fun = (train_objval + valid_objval + holdout_objval)/3
    listtotal_obj_fun.append(total_obj_fun)  


    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Elapsed time 5 : {elapsed_time : .2f} seconds")





In [107]:
    #Check condition 
    metconditon = 1
    if metconditon:
        within_sd_list.append(SeedNum) 
    else: 
        outof_sd_list.append(SeedNum) 
    #calculate train, valid, holdout Objective function
    #print(df_a[tv_split_colname].value_counts())

KeyboardInterrupt: 

In [ ]:
holdout_df_a_subset_flattened

In [ ]:
 #merge with pp, ee db 
    train_df_a = pd.merge(train_df_a, df1, on = 'VIN_Date', how = 'left')
    train_df_a_subset_flattened = generate_row(train_df_a)
    train_df_a_pp = train_df_a_subset_flattened[pp_columns].T.reset_index().drop(['index'], axis=1)
    #send to objective function 
    train_objval = objectivefunction(train_df_a_pp,stat_file)


In [ ]:
pp_columns

In [ ]:
#Read feather file columns
if 0:
    table = feather.read_table(full_feather_file_path)
    schema = table.schema
    col_names = schema.names
    df_columns = pd.DataFrame(col_names, columns = ['Column Names'])
    df_columns.to_csv(columns_file_path,index = False)
    df_columns.shape
    df_columns.nunique()
    duplicates = df_columns[df_columns.duplicated(keep= False)]
    print(duplicates)   


In [ ]:
#columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','ee_um_uim','ee_tot','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp','incurred_raw_med','incurred_raw_um_uim','incurred_raw_tot']
columns_to_load = ['VIN','Date','ee_bi','ee_pd','ee_pip','ee_med','ee_coll','ee_comp','vc_Vehicle_Type','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_med','incurred_raw_coll','incurred_raw_comp','claim_count_bi', 'claim_count_pd', 'claim_count_pip', 'claim_count_med','claim_count_coll', 'claim_count_comp']

In [ ]:
df1 = pd.read_feather(full_feather_file_path, columns =  columns_to_load )

In [ ]:
df1['Date'] = pd.to_datetime(df1['Date'])
df1['DateYMD'] = df1['Date'].dt.strftime('%Y%m%d')
df1['VIN_Date'] = df1['VIN'] +'_' +  df1['DateYMD']

In [ ]:
df1.shape

In [ ]:
df1.head()

In [ ]:
df1 = pd.merge(df1, df_a, on = 'VIN_Date', how = 'left')

In [ ]:
df1['VIN_Date'].nunique()

In [ ]:
df1.shape

In [ ]:
df_a['VIN_Date'].nunique()

In [ ]:
df1.shape

In [ ]:
def generate_row(df_subset):
    #np.random.seed(seed_num)
    
    agg_columns = ['ee_bi', 'ee_pd', 'ee_pip', 'ee_med', 'ee_coll', 'ee_comp',
       'claim_count_bi', 'claim_count_pd', 'claim_count_pip',
       'claim_count_coll', 'claim_count_comp', 'claim_count_med','incurred_raw_bi',
       'incurred_raw_pd', 'incurred_raw_pip', 'incurred_raw_coll',
       'incurred_raw_comp', 'incurred_raw_med'] 
    agg_dict = {col: 'sum' for col in agg_columns}
    df_subset_grouped = df_subset.groupby(['vc_Vehicle_Type']).agg(agg_dict).reset_index()

    #df_subset_grouped.head()
    df_subset_grouped['pp_bi'] = df_subset_grouped['incurred_raw_bi']/df_subset_grouped['ee_bi'] 
    df_subset_grouped['pp_pd'] = df_subset_grouped['incurred_raw_pd']/df_subset_grouped['ee_pd'] 
    df_subset_grouped['pp_pip'] = df_subset_grouped['incurred_raw_pip']/df_subset_grouped['ee_pip'] 
    df_subset_grouped['pp_med'] = df_subset_grouped['incurred_raw_med']/df_subset_grouped['ee_med'] 
    df_subset_grouped['pp_coll'] = df_subset_grouped['incurred_raw_coll']/df_subset_grouped['ee_coll'] 
    df_subset_grouped['pp_comp'] = df_subset_grouped['incurred_raw_comp']/df_subset_grouped['ee_comp'] 

    df_subset_grouped.set_index('vc_Vehicle_Type', inplace=True)
    flattened_columns = [f'{idx}_{col}' for idx in df_subset_grouped.index for col in df_subset_grouped.columns]
    flattened_data = df_subset_grouped.values.flatten()

    df_subset_flattened = pd.DataFrame([flattened_data], columns=flattened_columns)
    return df_subset_flattened

In [ ]:
df1_group_flattened = generate_row(df1)

In [ ]:
from itertools import chain 
ee_columns = [column for column in df1_group_flattened.columns if 'ee' in column]
pp_columns = [column for column in df1_group_flattened.columns if 'pp' in column]
all_columns = list(chain(pp_columns,ee_columns))
print(all_columns)

In [ ]:
csv_stat_file = 'df2_1_stat_pp_N500.csv'
stat_file = pd.read_csv(os.path.join(base_path, csv_stat_file))
stat_file.head()

In [ ]:
stat_file.index = stat_file['Unnamed: 0']

In [ ]:
stat_file.drop(columns = 'Unnamed: 0')

In [ ]:
df1_24D = df1_group_flattened[all_columns]

In [ ]:
df1_24D.head()

In [ ]:
df2_1_stat_pp_N500.csv

In [ ]:
df_d = pd.read_csv(os.path.join(fast_eda_path, 'df_design.csv'))

In [ ]:
np.max(df_a['superpolicy_id'])

In [ ]:
Num_unique = df_a['superpolicy_id'].nunique()


In [ ]:
unique_values = df_a['superpolicy_id'].unique()

In [ ]:


start_time = time.time()
#vSeedNum = 2#42
for SeedNum in range(1,6):
    tv_split_colname = 'tv_split_'+str(SeedNum)
    TrainPercentage = 0.6
    ValidPercentage = 0.2

    np.random.seed(SeedNum)  
    np.random.shuffle(unique_values)

    total_unique = len(unique_values)
    train_end = int(total_unique * TrainPercentage)
    val_end = train_end + int(total_unique * ValidPercentage)

    train_values = unique_values[:train_end]
    val_values = unique_values[train_end:val_end]
    holdout_values = unique_values[val_end:]

    value_to_split = {}
    for value in train_values:
        value_to_split[value] = 'train'
    for value in val_values:
        value_to_split[value] = 'valid'
    for value in holdout_values:
        value_to_split[value] = 'holdout'

    df_a[tv_split_colname] = df_a['superpolicy_id'].map(value_to_split)

    print(df_a[tv_split_colname].value_counts())

    end_time = time.time()
    elapsed_time = end_time - start_time

print(f"Elapsed time : {elapsed_time : .2f} seconds")


In [ ]:
df_a.shape

In [ ]:
df_a.columns

In [ ]:
df_a.drop('tv_split_5',axis = 1, inplace = True)

In [ ]:
df_a.columns

In [ ]:
df_a.head(10)

In [ ]:
if 0: 
    table = feather.read_table(feather_file_path)
    schema = table.schema
    col_names = schema.names
    df_columns = pd.DataFrame(col_names, columns = ['Column Names'])
    df_columns.to_csv(columns_file_path,index = False)

In [ ]:
# columns_to_load = ['ee_bi','ee_pd', 'ee_pip', 'ee_med','ee_coll','ee_comp','claim_count_bi','claim_count_pd','claim_count_pip','claim_count_coll','claim_count_comp','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_coll','incurred_raw_comp','incurred_raw_med','claim_count_med','VIN','Date','vc_Vehicle_Type','st']
columns_to_load = ['ee_bi','ee_pd', 'ee_pip', 'ee_med','ee_coll','ee_comp','incurred_raw_bi','incurred_raw_pd','incurred_raw_pip','incurred_raw_coll','incurred_raw_comp','incurred_raw_med','VIN','Date','vc_Vehicle_Type']
df2 = pd.read_feather(full_feather_file_path , columns = columns_to_load) 

In [ ]:
df2.shape

In [ ]:
df2_1_superpolicy = pd.merge(df2,df_a, on = ['VIN','Date'], how = 'left')

In [ ]:
df2_1_superpolicy.shape

In [ ]:
(df2_1_superpolicy.shape[0] - df2.shape[0])/df2_1_superpolicy.shape[0]

In [ ]:
df2_1_superpolicy.head()

In [ ]:
df2_1_superpolicy.columns

In [ ]:

df2_1_superpolicy.to_csv(file_df2_1_superpolicy,index = False )

In [ ]:
df1_1_superpolicy = pd.merge(df1_1,df_a, on = ['ID'], how = 'inner')

In [ ]:
column_names = df1.columns.tolist()
columns_df = pd.DataFrame(column_names, columns = ['Column Names'] )
columns_df.to_csv(columns_file_path,index = False)

In [ ]:
df1['ID'] = np.arange(1,len(df1)+1)
df1.shape

In [ ]:
df1.columns

In [ ]:
df2 = pd.merge(df_a,df1,on = 'ID',how = 'left' )

In [ ]:
columns_to_write = ['ID','pol','VIN','Date',]

In [ ]:
df2 = df1[columns_to_write]

In [ ]:
df2.head()

In [ ]:
#df1.head(1000).to_csv(csv_1000row_file_path,index = False)

In [ ]:
df2.to_csv(csv_file_path, index = False)

In [ ]:
df2.to_csv(csv_allrow_file_path, index = False) 

In [ ]:
outfile = 'R:/Projects/Carfax/VIP models/2023/Symbol/2023 Data/Data/glm_data_step2_final_specificolumns.csv'
df3 = pd.read_csv(outfile)

In [ ]:
df3.head()